# 🦾 Training Toolkit: Fine-tune PaliGemma for JSON Output

- **Input**: image, text prompt saying "extract JSON"
- **Output**: JSON object with information extracted from the image

In [ ]:
from dotenv import load_dotenv  
from pathlib import Path
import sys

sys.path.append(Path("..").resolve().as_posix())  # add the root of the project to the path to enable imports
_ = load_dotenv() # load HF token to access PaliGemma's gated repo

## Easy mode 🙈

**To get started**, convert your dataset to HF 🤗 Datasets format and save it locally.

It needs to have two columns:

- `image` column that is cast to `datasets.Image()` (see HF docs).
- `json` column that contains a **string** representation of your target JSON.


In [ ]:
DATASET_PATH = "path/to/dataset"

In [ ]:
from training_toolkit import build_trainer, paligemma_image_preset, image_json_preset

In [ ]:
trainer = build_trainer(
    **paligemma_image_preset.as_kwargs(),
    **image_json_preset.with_path(DATASET_PATH).as_kwargs(),
)

In [ ]:
trainer.train()

## Advanced mode 🙉

Preset parameters don't quite fit your case? Doesn't mean you need to build *everything* from scratch.

Learn how to combine Training Toolkit with custom tools to minimize time and energy required to train your adapter.

### 1. Tune preset parameters

Both `paligemma_image_preset` and `image_json_preset` are Pydantic dataclasses.

- Both return a dict when you call `as_kwargs()` on them.
- Combined, those dicts contain all parameters necessary to train an adapter.
- You can edit those parameters directly as dataclass attributes.

In [ ]:
from training_toolkit import build_trainer, paligemma_image_preset, image_json_preset

In [ ]:
# let's take a look at the arguments build_trainer expects

?build_trainer

In [ ]:
# we can find all these argumets in the presets

print(f"model preset: {paligemma_image_preset.as_kwargs().keys()}")
print(f"data preset args: {image_json_preset.with_path("path/to/dataset").as_kwargs().keys()}")

In [ ]:
# let's edit a couple hyperparameters 

paligemma_image_preset.hf_model_id="google/paligemma-3b-mix-448"

paligemma_image_preset.training_args["per_device_train_batch_size"] = 8
paligemma_image_preset.training_args["per_device_eval_batch_size"] = 8
paligemma_image_preset.training_args["eval_strategy"] = "no"
paligemma_image_preset.training_args["num_train_epochs"] = 10

In [ ]:
# now as_kwargs will return updated arguments

paligemma_image_preset.as_kwargs()

### 2. Customize dataset format

Say your dataset doesn't match the strict criteria described above. Or you have your own idea of what exactly the inputs and the targets should look like.

In order to use your dataset with the rest of the toolkit, you need to write **your own data collator**. This is a utility that turns a list of individual samples from your dataset into a batch ready to go into the model. Those batches are complete with padded tokenized text, attention masks and preprocessed images.

The HF 🤗 Transformers processor (see HF docs) is going to do the gnarly part. We need to do the following:

1. Pluck images out of the dataset
2. Prepare text inputs (aka prompts or prefixes)
3. Prepare target outputs (suffixes) - bits we want to teach the model to generate
4. Feed those things into the processor
5. Return the result

In [ ]:
from training_toolkit import DataPreset  # base class for data presets


JSON_PROMPT = "extract JSON."


class SpecialImageJSONCollator:

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        images = [example["image"] for example in examples]
        texts = [JSON_PROMPT for _ in range(len(examples))]  # input prompts
        labels = [example["special_json"] for example in examples]  # target JSONs

        batch = self.processor(
            text=texts,
            images=images,
            suffix=labels,  # PaliGemma calls targets "suffix"
            return_tensors="pt",
            padding="longest",  # for training it's better to pad to the right
        )
        return batch


# let's wrap the new collator into a custom DataPreset
special_image_json_preset = DataPreset(
    train_test_split=0.1,
    collator_cls=SpecialImageJSONCollator,
)

# finally, we can make sure everything loads correctly
special_image_json_preset.with_path("path/to/data").as_kwargs()